# v2b_syndata exploration (static)

This is a static walkthrough generated against the three committed example scenarios. Live frontend coming separately.

Scenarios used:
- `S01_baseline` — Nashville, mid-season, default population.
- `S_clim_miami_summer` — same population, Miami / July.
- `S_eq_bi` — baseline + bidirectional chargers.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science', 'no-latex'])

ROOT = Path('../data/example_scenarios')
scenarios = {name: {
    csv.stem: pd.read_csv(csv) for csv in (ROOT / name).glob('*.csv')
} for name in ('S01_baseline', 'S_clim_miami_summer', 'S_eq_bi')}

list(scenarios.keys())

## CSV row counts per scenario

Each scenario emits the same seven CSVs. Row counts vary with session intensity, climate (which extends/shortens HVAC tails), and DR program activity.

In [ ]:
rows = pd.DataFrame({
    name: {csv: len(df) for csv, df in csvs.items()}
    for name, csvs in scenarios.items()
}).fillna(0).astype(int)
rows

## Building load comparison

One-day load profile (flex + inflex). Miami summer shows the expected HVAC-driven flex amplification.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, (name, csvs) in zip(axes, scenarios.items()):
    df = csvs['building_load'].copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    one_day = df[df['datetime'] < df['datetime'].iloc[0] + pd.Timedelta(days=1)]
    ax.plot(one_day['datetime'], one_day['power_flex_kw'], label='flex', lw=1.2)
    ax.plot(one_day['datetime'], one_day['power_inflex_kw'], label='inflex', lw=1.2)
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Power (kW)')
plt.tight_layout()
plt.show()

## Session arrival distributions

Histogram of arrival hour-of-day across the simulation window. Baseline and bidirectional share the same population and ψ; Miami diverges due to climate-driven schedule shifts.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, csvs in scenarios.items():
    s = pd.to_datetime(csvs['sessions']['arrival'])
    ax.hist(s.dt.hour, bins=24, range=(0, 24), alpha=0.45, label=name)
ax.set_xlabel('Arrival hour of day')
ax.set_ylabel('Session count')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Manifest inspection (knob resolution)

Each scenario ships with a `manifest.json` that pins the resolved knob surface, seed lineage, per-CSV SHA-256, and the E5 (CONSENT) block.

In [ ]:
with open(ROOT / 'S01_baseline' / 'manifest.json') as f:
    manifest = json.load(f)

print('scenario_id:', manifest.get('scenario_id'))
print('seed:', manifest.get('seed'))
print()
print('--- 5 example knob entries ---')
knobs = manifest.get('resolved_knobs') or manifest.get('knobs') or {}
if isinstance(knobs, dict):
    for k, v in list(knobs.items())[:5]:
        print(f'  {k}: {v}')
print()
print('--- csv_sha256 ---')
shas = manifest.get('csv_sha256') or manifest.get('csv_hashes') or {}
for k, v in (shas.items() if isinstance(shas, dict) else []):
    print(f'  {k}: {v[:16]}...')
print()
print('--- E5 block ---')
e5 = manifest.get('e5') or manifest.get('consent') or {}
print(json.dumps(e5, indent=2)[:400])

## Comparing knob resolution across scenarios

Loading the three manifests and printing the resolved values that *differ* across scenarios. This is the canonical way to inspect what makes one scenario different from another.

In [ ]:
manifests = {}
for name in scenarios:
    with open(ROOT / name / 'manifest.json') as f:
        manifests[name] = json.load(f)

def flatten(d, prefix=''):
    out = {}
    for k, v in d.items():
        key = f'{prefix}.{k}' if prefix else k
        if isinstance(v, dict):
            out.update(flatten(v, key))
        else:
            out[key] = v
    return out

flat = {n: flatten(m.get('resolved_knobs') or m.get('knobs') or m) for n, m in manifests.items()}
all_keys = sorted(set().union(*[set(f.keys()) for f in flat.values()]))
diffs = []
for k in all_keys:
    vals = [str(flat[n].get(k)) for n in scenarios]
    if len(set(vals)) > 1:
        diffs.append({'knob': k, **{n: flat[n].get(k) for n in scenarios}})

diff_df = pd.DataFrame(diffs)
diff_df.head(30)

## Next: live frontend

This notebook is read-only and built against three pre-generated scenarios. A live Flask frontend that lets you sweep knobs interactively, regenerate scenarios, and diff them in-browser is coming separately.